# FullPokemon Testing

Tests for the calculators in the FullPokemon class

In [1]:
# Settings for paths to get data
from pathlib import Path
import sys

# getting the project root
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "summer26-pokemon-battle-predictor" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent


# finding the data folder
DATA_DIR = PROJECT_ROOT / "data"
# finding the csv of TRAINING data
PARSED_CSV_PATH = DATA_DIR / "data_cleaned.csv.zip"
# Adds tools to the system path so we can import the battle and full_pokemon modules
sys.path.append(str(PROJECT_ROOT))
from tools.full_pokemon import *

# replays
num_training_zips = 3
REPLAY_ROOT = DATA_DIR / "replays"
zip_paths = [REPLAY_ROOT / f"gen9randombattles_{i}.zip" for i in range(1,num_training_zips+1)]
from zipfile import ZipFile
replay_zips = [ZipFile(zip_path,'r') for zip_path in zip_paths]
import json

In [2]:
desired_mon_names = ['Weavile','Salamence','Haxorus','Corviknight','Chien-Pao','Conkeldurr','Fezandipiti','Banette','Wigglytuff','Sinistcha','Swampert','Lanturn','Toxtricity','Ditto','Palafin','Terapagos','Regigigas']
desired_item_names = ["Light Ball","Choice Band","Choice Specs","Choice Scarf","Eviolite","Assault Vest","Soul Dew"]
desired_team_dict = {}

battle_bytes_list = [replay_zip.read(file_name) for replay_zip in replay_zips for file_name in replay_zip.namelist()]

for battle_bytes in battle_bytes_list:
    try:
        battle = json.loads(battle_bytes)
        for team_index in range(2):
            for mon in battle["teams_full"][team_index].keys():
                mon_is_desired = (mon in desired_mon_names)
                item_is_desired = (battle["teams_full"][team_index][mon]["item"] in desired_item_names)
                if mon_is_desired or item_is_desired:
                    desired_team_dict[mon] = battle["teams_full"][team_index][mon]
                    if mon_is_desired:
                        desired_mon_names.remove(mon)
                    if item_is_desired:
                        desired_item_names.remove(battle["teams_full"][team_index][mon]["item"])
    except:
        continue
desired_mon_names = ['Weavile','Salamence','Haxorus','Corviknight','Chien-Pao','Conkeldurr','Fezandipiti','Banette','Wigglytuff','Sinistcha','Swampert','Lanturn','Toxtricity','Ditto','Palafin','Terapagos','Regigigas']
desired_item_names = ["Light Ball","Choice Band","Choice Specs","Choice Scarf","Eviolite","Assault Vest","Soul Dew"]

test_pokemon = {mon:FullPokemon(desired_team_dict[mon]) for mon in desired_team_dict.keys()}

In [3]:
print("Testing Individual Pokemon")
assert(test_pokemon["Palafin"].stats["atk"] == 291), "Palafin failed"
assert(test_pokemon["Terapagos"].stats["spd"] == 214), "Terapagos failed"
assert(test_pokemon["Regigigas"].stat_multiplier("atk") == 0.5), "Regigigas atk failed"
assert(test_pokemon["Regigigas"].stat_multiplier("spa") == 1), "Regigigas spa failed"
print("All is well.")

print()
print("Testing items")
assert(test_pokemon["Kyogre"].stat_multiplier("spe") == 1.5), "Kyogre (Choice Scarf) spe failed"
assert(test_pokemon["Kyogre"].stat_multiplier("spa") == 1), "Kyogre (Choice Scarf) spa failed"
assert(test_pokemon["Greninja"].stat_multiplier("spa") == 1.5), "Greninja (Choice Specs) spa failed"
assert(test_pokemon["Greninja"].stat_multiplier("spe") == 1), "Greninja (Choice Specs) spe failed"
assert(test_pokemon["Azumarill"].stat_multiplier("atk") == 3), "Azumarill (Choice Band) atk failed"
assert(test_pokemon["Azumarill"].stat_multiplier("spe") == 1), "Azumarill (Choice Band) spe failed"
assert(test_pokemon["Banette"].damage_multiplier() == 5324/4096), "Banette (Life Orb) failed"
assert(test_pokemon["Pikachu"].stat_multiplier("spa") == 2), "Pikachu (Light Ball) spa failed"
assert(test_pokemon["Pikachu"].stat_multiplier("def") == 1), "Pikachu (Light Ball) def failed"
assert(test_pokemon["Chansey"].stat_multiplier("def") == 1.5), "Chansey (Eviolite) def failed"
assert(test_pokemon["Chansey"].stat_multiplier("spd") == 1.5), "Chansey (Eviolite) spd failed"
assert(test_pokemon["Chansey"].stat_multiplier("spe") == 1), "Chansey (Eviolite) spe failed"
assert(test_pokemon["Hoopa"].stat_multiplier("spd") == 1.5), "Hoopa (Assault Vest) spd failed"
assert(test_pokemon["Hoopa"].stat_multiplier("def") == 1), "Hoopa (Assault Vest) def failed"
assert(test_pokemon["Latias"].damage_multiplier() == 4915/4096), "Latias (Soul Dew) failed"
print("All is well.")

print()
print("Now testing type_multiplier")
assert(FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Salamence']) == 6),"Weavile and Salamence failed"
assert(FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Haxorus']) == 3), "Weavile and Haxorus failed"
assert(FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Corviknight']) == 1.5), "Weavile and Corviknight failed"
assert(FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Chien-Pao']) == 0.75), "Weavile and Chien-Pao failed"
assert(FullPokemon.type_multiplier(test_pokemon['Conkeldurr'],test_pokemon['Fezandipiti']) == 1/2), "Conkeldurr and Fezandipiti failed"
assert(FullPokemon.type_multiplier(test_pokemon['Banette'],test_pokemon['Wigglytuff']) == 1/2), "Banette and Wigglytuff failed"
print("All is well.")

print()
print("Now testing one_v_one_damage")
ovo = FullPokemon.one_v_one_damage(test_pokemon['Weavile'],test_pokemon['Salamence'])
assert(ovo[0] >= 1 and ovo[1] == 0), "Weavile and Salamence failed"
ovo = FullPokemon.one_v_one_damage(test_pokemon['Sinistcha'],test_pokemon['Weavile'])
assert(ovo[0] > 0 and ovo[0] < 1 and ovo[1] >= 1), "Sinistcha and Weavile failed"
ovo = FullPokemon.one_v_one_damage(test_pokemon['Weavile'],test_pokemon['Conkeldurr'])
assert(ovo[0] > 0 and ovo[0] < 1 and ovo[1] >=1),"Weavile and Conkeldurr failed"
ovo = FullPokemon.one_v_one_damage(test_pokemon['Swampert'],test_pokemon['Corviknight'])
assert(ovo[0] >= 1 and ovo[1] > 0 and ovo[1] < 1), "Swampert and Corviknight failed"
# ad hoc forcing Toxtricity to carry choice specs
test_pokemon["Toxtricity"].item = "Choice Specs"
ovo = FullPokemon.one_v_one_damage(test_pokemon["Lanturn"],test_pokemon["Toxtricity"])
assert(ovo[0] >= 1 and ovo[1] >= 1), "Lanturn and Toxtricity failed"
print("All is well.")

print()
print("Now testing Ditto")
ditto_dam_to_ban = FullPokemon.damage(test_pokemon["Ditto"],test_pokemon["Banette"])
ban_dam_to_ditto = FullPokemon.damage(test_pokemon["Banette"],test_pokemon["Ditto"])
ovo = FullPokemon.one_v_one_damage(test_pokemon["Ditto"],test_pokemon["Banette"])
assert(ditto_dam_to_ban > 0.84 and ditto_dam_to_ban < 1.01), "Ditto damage to Banette failed"
assert(ban_dam_to_ditto > 1.42 and ban_dam_to_ditto < 1.68), "Banette damage to Ditto failed"
assert(ovo[0] > 0.84 and ovo[0] < 1.01 and ovo[1] > 1), "Ditto and Banette failed"
print("All is well.")

Testing Individual Pokemon
All is well.

Testing items
All is well.

Now testing type_multiplier
All is well.

Now testing one_v_one_damage
All is well.

Now testing Ditto
All is well.
